# Read the data from table and put it in JSON file

### Prompt used
```
You are a Python data-engineering assistant specializing in medical
data analysis and JSON reporting.

## Context
I have a SQLite database `reports.db` containing a table
`documents_relevant` with the following columns:

  filename, patient_name,
  calcium,  calcium_range,
  potassium, potassium_range,
  sodium,    sodium_range

The value columns (calcium, potassium, sodium) contain strings like
"9.7 mg/dl" or "145.0 mmol/L", or NULL if the test was not performed.
The range columns contain strings like "8.7 - 10.4" or "136 - 145".

## Task
Read every row from `documents_relevant` and apply the following logic
to each of the three analytes — calcium, potassium, and sodium:

  1. If the value column is NULL → skip that analyte entirely.
     Do not add anything to comments for it.
  2. If the numeric value is BELOW the lower bound of the range column
     → append to comments: "<Analyte>: <value> is LOW (normal range: <range>)"
  3. If the numeric value is ABOVE the upper bound of the range column
     → append to comments: "<Analyte>: <value> is HIGH (normal range: <range>)"
  4. If the numeric value is WITHIN the range (inclusive)
     → append to comments: "<Analyte>: <value> is NORMAL (normal range: <range>)"

Collect all analyte comments for a row into a single "comments" field
(a JSON array of strings).

## Output
Write all rows to a file named `results.json` as a top-level JSON array.
Each element must follow this exact structure:

{
  "filename":         "<string>",
  "patient_name":     "<string>",
  "calcium":          "<string or null>",
  "calcium_range":    "<string or null>",
  "potassium":        "<string or null>",
  "potassium_range":  "<string or null>",
  "sodium":           "<string or null>",
  "sodium_range":     "<string or null>",
  "comments": [
    "Calcium: 9.7 mg/dl is NORMAL (normal range: 8.7 - 10.4)",
    "Potassium: 5.03 mmol/L is LOW (normal range: 3.5 - 5.1)",
    "Sodium: 148.0 mmol/L is HIGH (normal range: 136 - 145)"
  ]
}

## Constraints
- Open `reports.db` in READ-ONLY mode using the SQLite URI:
  sqlite3.connect("file:reports.db?mode=ro", uri=True)
- Do NOT create, modify, or write to any database table.
- Do NOT create any intermediate files — only `results.json`.
- Use Python's `re` module to parse numeric values and reference ranges.
- Handle both hyphen (-) and en-dash (–) as range separators.
- Print a per-row console summary showing filename, patient name,
  and each comment on its own line.

## Sample Input Row
filename       : rep1.pdf
patient_name   : John Doe4
calcium        : 9.7 mg/dl
calcium_range  : 8.7 - 10.4
potassium      : 5.03 mmol/L
potassium_range: 3.5 - 5.1
sodium         : 145.0 mmol/L
sodium_range   : 136 - 145

## Expected JSON Output for Sample Row
{
  "filename": "rep1.pdf",
  "patient_name": "John Doe4",
  "calcium": "9.7 mg/dl",
  "calcium_range": "8.7 - 10.4",
  "potassium": "5.03 mmol/L",
  "potassium_range": "3.5 - 5.1",
  "sodium": "145.0 mmol/L",
  "sodium_range": "136 - 145",
  "comments": [
    "Calcium: 9.7 mg/dl is NORMAL (normal range: 8.7 - 10.4)",
    "Potassium: 5.03 mmol/L is NORMAL (normal range: 3.5 - 5.1)",
    "Sodium: 145.0 mmol/L is NORMAL (normal range: 136 - 145)"
  ]
}

## Expected Console Output for Sample Row
  file='rep1.pdf'  patient='John Doe4'
    → Calcium: 9.7 mg/dl is NORMAL (normal range: 8.7 - 10.4)
    → Potassium: 5.03 mmol/L is NORMAL (normal range: 3.5 - 5.1)
    → Sodium: 145.0 mmol/L is NORMAL (normal range: 136 - 145)
```

In [3]:
"""
analyse_results.py
------------------
Reads `documents_relevant` table from reports.db (READ ONLY — no DB writes).

For each row:
  - Calcium, Potassium, Sodium: if NULL → skipped.
  - If value is out of reference range → flagged LOW or HIGH in comments.
  - If value is within reference range → noted as NORMAL in comments.
  - All analyte comments collected into a single "comments" list per row.

Output: results.json
"""

import sqlite3
import re
import json
from pathlib import Path

DB_PATH     = "reports.db"
SOURCE_TABLE = "documents_relevant"
OUTPUT_JSON  = "results.json"


# ── Helpers ───────────────────────────────────────────────────────────────────

def parse_range(range_str: str) -> tuple[float, float] | tuple[None, None]:
    """'8.7 - 10.4'  →  (8.7, 10.4)"""
    if not range_str:
        return None, None
    m = re.search(r'([\d.]+)\s*[-–]\s*([\d.]+)', range_str.strip())
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)


def extract_numeric(value_str: str) -> float | None:
    """'9.7 mg/dl'  →  9.7"""
    if not value_str:
        return None
    m = re.match(r'([\d.]+)', value_str.strip())
    return float(m.group(1)) if m else None


def build_comment(name: str, value_str: str | None, range_str: str | None) -> str | None:
    """Return comment string or None if value is NULL."""
    value = extract_numeric(value_str)
    if value is None:
        return None                         # NULL → skip silently
    low, high = parse_range(range_str)
    if low is None or high is None:
        return f"{name}: {value_str} (reference range unavailable)"
    if value < low:
        status = "LOW"
    elif value > high:
        status = "HIGH"
    else:
        status = "NORMAL"
    return f"{name}: {value_str} is {status} (normal range: {range_str})"


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    if not Path(DB_PATH).exists():
        print(f"ERROR: '{DB_PATH}' not found.")
        return

    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(f"""
        SELECT filename, patient_name,
               calcium,   calcium_range,
               potassium, potassium_range,
               sodium,    sodium_range
        FROM {SOURCE_TABLE}
    """).fetchall()
    conn.close()

    print(f"Read {len(rows)} row(s) from '{SOURCE_TABLE}' in {DB_PATH}\n")

    output = []

    for row in rows:
        comments = []
        for name, val_col, range_col in [
            ("Calcium",   row["calcium"],   row["calcium_range"]),
            ("Potassium", row["potassium"], row["potassium_range"]),
            ("Sodium",    row["sodium"],    row["sodium_range"]),
        ]:
            comment = build_comment(name, val_col, range_col)
            if comment:
                comments.append(comment)

        record = {
            "filename":         row["filename"],
            "patient_name":     row["patient_name"],
            "calcium":          row["calcium"],
            "calcium_range":    row["calcium_range"],
            "potassium":        row["potassium"],
            "potassium_range":  row["potassium_range"],
            "sodium":           row["sodium"],
            "sodium_range":     row["sodium_range"],
            "comments":         comments,
        }
        output.append(record)

        print(f"  file='{row['filename']}'  patient='{row['patient_name']}'")
        if comments:
            for c in comments:
                print(f"    → {c}")
        else:
            print("    → (all values NULL — no comments)")
        print()

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"Done. {len(output)} record(s) written to '{OUTPUT_JSON}'.")


if __name__ == "__main__":
    main()

Read 2 row(s) from 'documents_relevant' in reports.db

  file='rep1.pdf'  patient='John Doe4'
    → Calcium: 9.7 mg/dl is NORMAL (normal range: 8.7 - 10.4)
    → Potassium: 5.03 mmol/L is NORMAL (normal range: 3.5 - 5.1)
    → Sodium: 145.0 mmol/L is NORMAL (normal range: 136 - 145)

  file='rep2.pdf'  patient='John Doe4'
    → (all values NULL — no comments)

Done. 2 record(s) written to 'results.json'.
